# IT549 Lab 2
### Name: Dhruv Patel
### ID: 202301095


In [1]:
import pandas as pd
import numpy as np
import re
import ast
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import mean_squared_error, f1_score, hamming_loss
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Using device: cuda


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data Preparation
Loading the dataset, keeping only the allowed columns (`genre`, `keywords`, `tagline`, `overview`, `voting_average`), and applying text preprocessing (lowercase, removing URLs/punctuation/numbers, tokenization, and lemmatization).

In [3]:
df = pd.read_csv('/content/drive/MyDrive/IT549_Lab/Lab2/movies.csv')

allowed_cols = ['genres', 'keywords', 'tagline', 'overview', 'vote_average']
df = df[allowed_cols].copy()

df.dropna(subset=['vote_average', 'genres'], inplace=True)

text_cols = ['overview', 'tagline', 'keywords']
for col in text_cols:
    df[col] = df[col].fillna('')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

for col in text_cols:
    df[f'clean_{col}'] = df[col].apply(preprocess_text)

def extract_names(item):
    try:
        parsed = ast.literal_eval(item)
        if isinstance(parsed, list):
            return [d['name'] for d in parsed if 'name' in d]
    except:
        pass

    if isinstance(item, str):
        item = item.replace('Science Fiction', 'Science_Fiction')
        item = item.replace('TV Movie', 'TV_Movie')

        if ',' in item:
            genres = [g.strip() for g in item.split(',')]
        else:
            genres = item.split()

        return [g.replace('_', ' ') for g in genres]

    return []

df['parsed_genres'] = df['genres'].apply(extract_names)
df = df[df['parsed_genres'].map(len) > 0]

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

Train size: 3342, Val size: 716, Test size: 717


## GloVe Embedding Pipeline
Loading 100D GloVe vectors, computing vocabulary coverage, and creating TF-IDF weighted document embeddings.

In [4]:
glove_path = '/content/drive/MyDrive/IT549_Lab/Lab2/glove.6B.100d.txt'
embeddings_dict = {}
with open(glove_path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_dict[word] = vector

embedding_dim = 100

def get_coverage(text_series, glove_dict):
    vocab = set()
    for text in text_series:
        vocab.update(text.split())

    covered = sum(1 for word in vocab if word in glove_dict)
    coverage_pct = (covered / len(vocab)) * 100 if len(vocab) > 0 else 0
    return coverage_pct, len(vocab)

cov_pct, total_vocab = get_coverage(train_df['clean_overview'], embeddings_dict)
print(f"Embedding Coverage (Overview): {cov_pct:.2f}% of {total_vocab} unique tokens.")

def compute_tfidf_weighted_embeddings(train_texts, val_texts, test_texts, glove_dict):
    vectorizer = TfidfVectorizer()
    vectorizer.fit(train_texts)

    def process(texts):
        tfidf_matrix = vectorizer.transform(texts)
        feature_names = vectorizer.get_feature_names_out()

        doc_embeddings = []
        for i in range(texts.shape[0]):
            doc_vec = np.zeros(embedding_dim)
            total_weight = 0.0

            _, cols = tfidf_matrix[i].nonzero()
            for col in cols:
                word = feature_names[col]
                if word in glove_dict:
                    weight = tfidf_matrix[i, col]
                    doc_vec += weight * glove_dict[word]
                    total_weight += weight

            if total_weight > 0:
                doc_vec /= total_weight
            doc_embeddings.append(doc_vec)
        return np.array(doc_embeddings)

    return process(train_texts), process(val_texts), process(test_texts)

X_train_over, X_val_over, X_test_over = compute_tfidf_weighted_embeddings(
    train_df['clean_overview'], val_df['clean_overview'], test_df['clean_overview'], embeddings_dict
)

X_train_tag, X_val_tag, X_test_tag = compute_tfidf_weighted_embeddings(
    train_df['clean_tagline'], val_df['clean_tagline'], test_df['clean_tagline'], embeddings_dict
)

y_train_reg = train_df['vote_average'].values
y_val_reg = val_df['vote_average'].values
y_test_reg = test_df['vote_average'].values

Embedding Coverage (Overview): 96.81% of 15633 unique tokens.


## Model A: Rating Prediction (Regression)
Training an MLP to predict `vote_average` from document embeddings, comparing `overview` vs `tagline` performance against a global mean baseline.

In [11]:
class RegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(RegressionModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

def train_and_eval_regression(X_train, X_test, y_train, y_test, title):
    print(f"\nTraining Model A: {title}")
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)

    dataset = TensorDataset(X_train_t, y_train_t)
    loader = DataLoader(dataset, batch_size=64, shuffle=True)

    model = RegressionModel(embedding_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)

    epochs = 15
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_b, y_b in loader:
            optimizer.zero_grad()
            preds = model(X_b)
            loss = criterion(preds, y_b)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_b.size(0)

        train_loss /= len(loader.dataset)

        model.eval()
        with torch.no_grad():
            test_preds = model(X_test_t)
            test_loss = criterion(test_preds, y_test_t).item()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | Train MSE: {train_loss:.4f} | Test MSE: {test_loss:.4f}")

    model.eval()
    with torch.no_grad():
        preds_test = model(X_test_t).cpu().numpy()

    mse = mean_squared_error(y_test, preds_test)
    rmse = np.sqrt(mse)
    return mse, rmse

# Baseline
mean_rating = np.mean(y_train_reg)
baseline_preds = np.full_like(y_test_reg, mean_rating)
baseline_mse = mean_squared_error(y_test_reg, baseline_preds)
baseline_rmse = np.sqrt(baseline_mse)

print(f"Baseline (Mean) -> MSE: {baseline_mse:.4f}, RMSE: {baseline_rmse:.4f}")

mse_over, rmse_over = train_and_eval_regression(X_train_over, X_test_over, y_train_reg, y_test_reg, "Overview")
print(f"Final Model A (Overview) -> MSE: {mse_over:.4f}, RMSE: {rmse_over:.4f}")

mse_tag, rmse_tag = train_and_eval_regression(X_train_tag, X_test_tag, y_train_reg, y_test_reg, "Tagline")
print(f"Final Model A (Tagline) -> MSE: {mse_tag:.4f}, RMSE: {rmse_tag:.4f}")

Baseline (Mean) -> MSE: 1.1165, RMSE: 1.0567

Training Model A: Overview
Epoch 01/15 | Train MSE: 13.0520 | Test MSE: 1.6998
Epoch 05/15 | Train MSE: 4.1730 | Test MSE: 1.4252
Epoch 10/15 | Train MSE: 3.4072 | Test MSE: 1.2071
Epoch 15/15 | Train MSE: 2.9454 | Test MSE: 1.3241
Final Model A (Overview) -> MSE: 1.3241, RMSE: 1.1507

Training Model A: Tagline
Epoch 01/15 | Train MSE: 14.4510 | Test MSE: 5.9887
Epoch 05/15 | Train MSE: 2.8514 | Test MSE: 1.3082
Epoch 10/15 | Train MSE: 2.4051 | Test MSE: 1.2822
Epoch 15/15 | Train MSE: 2.2943 | Test MSE: 1.1950
Final Model A (Tagline) -> MSE: 1.1950, RMSE: 1.0931


## Model B: Genre Prediction (Multi-Label Classification)
Using PyTorch with `BCEWithLogitsLoss` to predict genres from the text embeddings.

In [10]:
mlb = MultiLabelBinarizer()
y_train_clf = mlb.fit_transform(train_df['parsed_genres'])
y_val_clf = mlb.transform(val_df['parsed_genres'])
y_test_clf = mlb.transform(test_df['parsed_genres'])
num_classes = len(mlb.classes_)

class ClassificationModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ClassificationModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.net(x)

def train_and_eval_classification(X_train, X_test, y_train, y_test, title):
    print(f"\nTraining Model B: {title}")
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).to(device)

    dataset = TensorDataset(X_train_t, y_train_t)
    loader = DataLoader(dataset, batch_size=64, shuffle=True)

    model = ClassificationModel(embedding_dim, num_classes).to(device)

    pos_counts = y_train.sum(axis=0)
    neg_counts = len(y_train) - pos_counts
    pos_weights = (neg_counts + 1e-5) / (pos_counts + 1e-5)

    pos_weights = np.clip(pos_weights, 1.0, 20.0)
    pos_weight_t = torch.tensor(pos_weights, dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    epochs = 20
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_b, y_b in loader:
            optimizer.zero_grad()
            preds = model(X_b)
            loss = criterion(preds, y_b)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_b.size(0)

        train_loss /= len(loader.dataset)

        model.eval()
        with torch.no_grad():
            logits_test = model(X_test_t)
            test_loss = criterion(logits_test, y_test_t).item()
            probs_test = torch.sigmoid(logits_test).cpu().numpy()
            preds_binary = (probs_test > 0.4).astype(int) # Slightly higher threshold
            micro = f1_score(y_test, preds_binary, average='micro')

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test Micro-F1: {micro:.4f}")

    model.eval()
    with torch.no_grad():
        logits = model(X_test_t).cpu()
        probs = torch.sigmoid(logits).numpy()
        preds_test = (probs > 0.4).astype(int)

    micro_f1 = f1_score(y_test, preds_test, average='micro')
    macro_f1 = f1_score(y_test, preds_test, average='macro', zero_division=0)
    h_loss = hamming_loss(y_test, preds_test)

    return micro_f1, macro_f1, h_loss

micro_o, macro_o, hl_o = train_and_eval_classification(X_train_over, X_test_over, y_train_clf, y_test_clf, "Overview")
print(f"Final Model B (Overview) -> Micro-F1: {micro_o:.4f}, Macro-F1: {macro_o:.4f}, Hamming Loss: {hl_o:.4f}")

micro_t, macro_t, hl_t = train_and_eval_classification(X_train_tag, X_test_tag, y_train_clf, y_test_clf, "Tagline")
print(f"Final Model B (Tagline) -> Micro-F1: {micro_t:.4f}, Macro-F1: {macro_t:.4f}, Hamming Loss: {hl_t:.4f}")


Training Model B: Overview
Epoch 01/20 | Train Loss: 0.9534 | Test Loss: 0.8366 | Test Micro-F1: 0.3667
Epoch 05/20 | Train Loss: 0.6570 | Test Loss: 0.6813 | Test Micro-F1: 0.4847
Epoch 10/20 | Train Loss: 0.5779 | Test Loss: 0.6783 | Test Micro-F1: 0.5055
Epoch 15/20 | Train Loss: 0.5352 | Test Loss: 0.6903 | Test Micro-F1: 0.5086
Epoch 20/20 | Train Loss: 0.5054 | Test Loss: 0.7233 | Test Micro-F1: 0.5184
Final Model B (Overview) -> Micro-F1: 0.5184, Macro-F1: 0.4008, Hamming Loss: 0.1907

Training Model B: Tagline
Epoch 01/20 | Train Loss: 1.0355 | Test Loss: 0.9712 | Test Micro-F1: 0.2901
Epoch 05/20 | Train Loss: 0.8804 | Test Loss: 0.9201 | Test Micro-F1: 0.3312
Epoch 10/20 | Train Loss: 0.8109 | Test Loss: 0.9293 | Test Micro-F1: 0.3405
Epoch 15/20 | Train Loss: 0.7690 | Test Loss: 0.9483 | Test Micro-F1: 0.3470
Epoch 20/20 | Train Loss: 0.7360 | Test Loss: 0.9934 | Test Micro-F1: 0.3479
Final Model B (Tagline) -> Micro-F1: 0.3479, Macro-F1: 0.2535, Hamming Loss: 0.3393


## Frequent & Genre-Indicative Words
Analyzing the `overview` text to extract the most/least frequent words per genre, and identifying indicative words using Logistic Regression coefficients trained on TF-IDF features.

In [9]:
cv = CountVectorizer(min_df=3)
X_counts = cv.fit_transform(train_df['clean_overview'])
vocab = np.array(cv.get_feature_names_out())

tfidf_vec = TfidfVectorizer(min_df=3)
X_tfidf = tfidf_vec.fit_transform(train_df['clean_overview'])
vocab_tfidf = np.array(tfidf_vec.get_feature_names_out())

top_genres = pd.Series([g for sublist in train_df['parsed_genres'] for g in sublist]).value_counts().head(5).index

print("Task 5 & 6 Results")
for genre in top_genres:
    genre_idx = [i for i, genres in enumerate(train_df['parsed_genres']) if genre in genres]

    if len(genre_idx) == 0:
        continue

    genre_counts = X_counts[genre_idx].sum(axis=0).A1
    nonzero_idx = genre_counts.nonzero()[0]

    sorted_idx = np.argsort(genre_counts[nonzero_idx])
    top_10 = vocab[nonzero_idx[sorted_idx][-10:]][::-1]
    bottom_10 = vocab[nonzero_idx[sorted_idx][:10]]

    y_genre = np.zeros(len(train_df))
    y_genre[genre_idx] = 1

    lr = LogisticRegression(max_iter=1000, class_weight='balanced')
    lr.fit(X_tfidf, y_genre)

    indicative_idx = np.argsort(lr.coef_[0])[-10:][::-1]
    indicative_words = vocab_tfidf[indicative_idx]

    print(f"\nGenre: {genre}")
    print(f"Top 10 Frequent: {', '.join(top_10)}")
    print(f"Bottom 10 Frequent (>=3): {', '.join(bottom_10)}")
    print(f"Top 10 Indicative (TF-IDF + LR): {', '.join(indicative_words)}")

Task 5 & 6 Results

Genre: Drama
Top 10 Frequent: life, find, story, young, new, one, year, man, family, love
Bottom 10 Frequent (>=3): fearing, fearsome, figuring, finn, wallace, warning, warns, waste, yates, feud
Top 10 Indicative (TF-IDF + LR): life, story, struggle, drama, wife, relationship, change, mother, husband, love

Genre: Comedy
Top 10 Frequent: life, friend, new, find, get, one, love, two, year, world
Bottom 10 Frequent (>=3): willis, wipe, wire, wisdom, hitler, hoffman, abuse, abusive, wonderland, worldwide
Top 10 Indicative (TF-IDF + LR): comedy, big, girlfriend, friend, guy, store, parent, single, buddy, head

Genre: Thriller
Top 10 Frequent: life, find, one, man, new, young, year, world, two, take
Bottom 10 Frequent (>=3): abandon, abby, abduct, aboard, abroad, absence, abuse, academic, everywhere, evolution
Top 10 Indicative (TF-IDF + LR): murder, agent, killer, assassin, thriller, suspect, kidnapped, crime, secret, event

Genre: Action
Top 10 Frequent: world, life, f